In [1]:
from utils import *
import pandas as pd
import os
import numpy as np

In [2]:
# load data 
file_path = os.path.join("tables", "exerciseTableForBN_python_processed.csv")
exerciseTable = pd.read_csv(file_path)

In [3]:
# all static variables: ['age', 'gender', 'diabetesDuration', 'BMI', 'HbA1c', 'InsSensitivity', 'InsCarbRatio']
# all pre exercise variables: ['preExerciseRoc', 'startExerciseGlucoseLevel', 'preExerciseGlucoseCV', 'IOBnorm', 'COBnorm', 'AOB', 'TotalCWL']
# all exercise variables: ['MET_min', 'ExerciseModality', 'DurationValue', 'MET']
# all outcome variables during exercise: ['exerciseMaxExcursion', 'exerciseMaxSpikeRoc', 'exerciseMaxDropRoc', 'exerciseNadir', 'exercisePeak', 'exerciseTIR', 'exerciseTBR', 'exerciseAUC70']
# all outcome variables post exercise: ['maxGlucosePostExercise', 'minGlucosePostExercise','postExerciseTimeToNadir', 'postExerciseTIR','postExerciseTBR','postExerciseTAR', 'postExerciseGlucoseCV', 'postExerciseAUC70']


# network_structure = 'complete'
# network_structure = 'lateHypoglycemia'
# network_structure = 'glycemicStability'
network_structure = 'acuteWorkout'

if network_structure == 'complete':
    tiers = {
        'static': ['age', 'gender', 'diabetesDuration', 'BMI', 'HbA1c', 'InsSensitivity', 'InsCarbRatio'],
        'pre': ['preExerciseRoc', 'startExerciseGlucoseLevel', 'preExerciseGlucoseCV', 'IOBnorm', 'COBnorm', 'AOB', 'TotalCWL'],
        'exercise' : ['MET', 'DurationValue', 'ExerciseModality'],
        #'exercise' : ['MET_min', 'ExerciseModality'],
        'outcome_during': ['exerciseMaxExcursion', 'exerciseMaxSpikeRoc', 'exerciseMaxDropRoc', 'exerciseNadir', 'exercisePeak', 'exerciseTIR', 'exerciseTBR', 'exerciseAUC70'],
        'outcome_post': ['maxGlucosePostExercise', 'minGlucosePostExercise','postExerciseTimeToNadir', 'postExerciseTIR','postExerciseTBR','postExerciseTAR', 'postExerciseGlucoseCV', 'postExerciseAUC70', 'postExerciseHypoEvent']
    }

if network_structure == 'lateHypoglycemia':
       tiers = {
        'static': ['age', 'gender', 'diabetesDuration', 'BMI', 'HbA1c', 'InsSensitivity', 'InsCarbRatio'],
        'pre': ['preExerciseRoc', 'startExerciseGlucoseLevel', 'preExerciseGlucoseCV', 'IOBnorm', 'COBnorm', 'AOB', 'TotalCWL'],
        'exercise' : ['MET', 'DurationValue', 'ExerciseModality'],
        #'exercise' : ['MET_min', 'ExerciseModality'],
        'outcome_during': ['exerciseMaxDropRoc', 'exerciseNadir'],
        'outcome_post': ['postExerciseHypoEvent']
    }
       
if network_structure == 'glycemicStability':
    tiers = {
        'static': ['age', 'gender', 'diabetesDuration', 'BMI', 'HbA1c', 'InsSensitivity', 'InsCarbRatio'],
        'pre': ['preExerciseRoc', 'startExerciseGlucoseLevel', 'preExerciseGlucoseCV', 'IOBnorm', 'COBnorm', 'AOB', 'TotalCWL'],
        'exercise' : ['MET', 'DurationValue', 'ExerciseModality'],
        #'exercise' : ['MET_min', 'ExerciseModality'],
        'outcome_during': ['exerciseTIR', 'exerciseMaxExcursion'],
        'outcome_post': ['postExerciseTIR','postExerciseGlucoseCV']
    }
    
if network_structure == 'acuteWorkout':
    tiers = {
        'static': ['age', 'gender', 'diabetesDuration', 'BMI', 'HbA1c', 'InsSensitivity', 'InsCarbRatio'],
        'pre': ['preExerciseRoc', 'startExerciseGlucoseLevel', 'preExerciseGlucoseCV', 'IOBnorm', 'COBnorm', 'AOB', 'TotalCWL'],
        'exercise' : ['MET', 'DurationValue', 'ExerciseModality'],
        #'exercise' : ['MET_min', 'ExerciseModality'],
        'outcome_during': ['exerciseMaxSpikeRoc', 'exerciseMaxDropRoc'],
        'outcome_post': []
    }

all_features = exerciseTable.columns.tolist()
features_to_drop = set(all_features) - set(sum(tiers.values(), []))

In [ ]:
# Discretize data

df = exerciseTable.copy()

df_discrete = discretize_data(df, 
    cv_strategy = "statistical",
    bmi_strategy = "clinical",
    hba1c_strategy = "clinical", 
    glucose_strategy = "clinical", 
    roc_strategy = "clinical",
    cols_to_remove = features_to_drop)

#Check Discretization
check_discretization(df_discrete) # comment if not needed!


# Collapse bins
df_discrete_collapsed = collapse_sparse_bins(df_discrete)

In [ ]:
# Visualize Feature Distribution after Discretization (and after bins collapse)

target_feature = 'postExerciseHypoEvent'

plot_single_feature_distribution(df_discrete, target_feature, title_prefix="- Clinical Thresholds")
plot_single_feature_distribution(df_discrete_collapsed, target_feature, title_prefix="- Clinical Thresholds After Merge")


In [ ]:
# Correlation Analysis

import seaborn as sns

exerciseTable_numeric = exerciseTable.copy()
exerciseTable_numeric['ExerciseModality'] = exerciseTable_numeric['ExerciseModality'].astype('category').cat.codes

exerciseTable_numeric_static = exerciseTable_numeric[tiers['static']]
exerciseTable_numeric_pre = exerciseTable_numeric[tiers['pre']]
exerciseTable_numeric_exercise = exerciseTable_numeric[tiers['exercise']]
exerciseTable_numeric_outcomes_during = exerciseTable_numeric[tiers['outcome_during']]
exerciseTable_numeric_outcomes_post = exerciseTable_numeric[tiers['outcome_post']]

corr_matrix_static = exerciseTable_numeric_static.corr()
corr_matrix_pre = exerciseTable_numeric_pre.corr()
corr_matrix_exercise = exerciseTable_numeric_exercise.corr()
corr_matrix_outcomes_during = exerciseTable_numeric_outcomes_during.corr()
corr_matrix_outcomes_post = exerciseTable_numeric_outcomes_post.corr()

%config InlineBackend.figure_format = 'retina'

plt.figure(figsize=(15, 10))
sns.heatmap(corr_matrix_static, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title("Correlation Matrix - Static Variables")

plt.figure(figsize=(15, 10))
sns.heatmap(corr_matrix_pre, annot=True, fmt=".2f", cmap='coolwarm', cbar=True) 
plt.title("Correlation Matrix - Pre-Exercise Variables")

plt.figure(figsize=(15, 10))
sns.heatmap(corr_matrix_exercise, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title("Correlation Matrix - Exercise Variables")

plt.figure(figsize=(15, 10))
sns.heatmap(corr_matrix_outcomes_during, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title("Correlation Matrix - Outcome Variables During Exercise")

if tiers['outcome_post']:
    plt.figure(figsize=(15, 10))
    sns.heatmap(corr_matrix_outcomes_post, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
    plt.title("Correlation Matrix - Outcome Variables Post Exercise")


In [ ]:
# Data Driven BN

collapse = True

if(collapse):
    learner_data_driven = gum.BNLearner(df_discrete_collapsed)
else:
    learner_data_driven = gum.BNLearner(df_discrete)

learner_data_driven.useGreedyHillClimbing()
bn_data_driven = learner_data_driven.learnBN()

visualize_network(bn_data_driven, tiers)

In [ ]:
# Constrained BN

collapse = True

if(collapse):
    learner_constrained = gum.BNLearner(df_discrete_collapsed)
else:
    learner_constrained = gum.BNLearner(df_discrete)
    
learner_constrained.useGreedyHillClimbing()
learner_constrained = apply_expert_constraints(learner_constrained, tiers)
bn_constrained = learner_constrained.learnBN()

visualize_network(bn_constrained, tiers)


FEATURE IMPORTANCE 

In [ ]:
import pyagrum as gum
import matplotlib.pyplot as plt

def plot_feature_importance(bn, target_node, threshold=0.001):
    """
    Calculates and plots the Mutual Information (Feature Importance) 
    for a given target node in a Bayesian Network.
    """
    # 1. Initialize the inference engine (Required to compute entropy)
    ie = gum.LazyPropagation(bn)
    
    # 2. Calculate Mutual Information for all nodes relative to the target
    mi_dict = {}
    for node in bn.names():
        if node != target_node:
            # FIX: Use the native InformationTheory class
            # Syntax: InformationTheory(inference_engine, target_X, target_Y)
            it = gum.InformationTheory(ie, target_node, node)
            mi = it.mutualInformationXY()
            mi_dict[node] = mi
            
    # 3. Sort the dictionary by importance (highest to lowest)
    sorted_mi = dict(sorted(mi_dict.items(), key=lambda item: item[1], reverse=True))
    
    # 4. Filter out features with near-zero importance to keep the chart clean
    filtered_mi = {k: v for k, v in sorted_mi.items() if v > threshold}
    
    if not filtered_mi:
        print(f"No features found with importance > {threshold}")
        return
        
    # 5. Prepare data for plotting 
    # (We reverse the lists so the highest value sits at the top of the horizontal chart)
    features = list(filtered_mi.keys())[::-1]
    importances = list(filtered_mi.values())[::-1]
    
    # 6. Generate the Plot using your Okabe-Ito Vermillion color
    plt.figure(figsize=(10, max(6, len(features) * 0.4)))
    bars = plt.barh(features, importances, color='#D55E00', edgecolor='black')
    
    plt.xlabel('Mutual Information (Bits / Entropy Reduction)', fontsize=12)
    plt.title(f'Feature Importance for: {target_node}', fontsize=14, fontweight='bold')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    
    # Clean layout for academic presentation
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()
    
    # 7. Print the exact numerical values so you can copy them into a thesis table
    print(f"--- Exact Mutual Information Scores for '{target_node}' ---")
    for feat, score in filtered_mi.items():
        print(f"{feat}: {score:.4f} bits")


# Use the function to plot feature importance for the target node of interest

if network_structure == 'lateHypoglycemia':
    plot_feature_importance(bn_constrained, target_node='postExerciseHypoEvent')

if network_structure == 'glycemicStability':
    plot_feature_importance(bn_constrained, target_node='postExerciseTIR')

if network_structure == 'acuteWorkout':
    plot_feature_importance(bn_constrained, target_node='exerciseMaxDropRoc')

INFERENCE

In [ ]:

if network_structure == 'lateHypoglycemia':
    interactive_inference_dashboard(
    bn = bn_constrained, 
    evidence_nodes = [ 'startExerciseGlucoseLevel', 'diabetesDuration', 'BMI', 'exerciseNadir'], 
    target_node = 'postExerciseHypoEvent'
)


if network_structure == 'glycemicStability':
    interactive_inference_dashboard(
    bn = bn_constrained, 
    evidence_nodes = [ 'startExerciseGlucoseLevel', 'exerciseTIR', 'HbA1c'], 
    target_node = 'postExerciseTIR'
)
    
if network_structure == 'acuteWorkout':
    interactive_inference_dashboard(
    bn = bn_constrained, 
    evidence_nodes = [ 'preExerciseRoc', 'DurationValue', 'InsSensitivity'], 
    target_node = 'exerciseMaxDropRoc'
)

